# Figures 4D, 4E and S01-S06, S09: dataset overview and cell-state atlas

Establishes the combinatorial NEPC screen: the UMAP coloured by manuscript state
(NEPC-N / NEPC-A&N / NEPC-A), time point and cell-cycle phase; trajectory
signature scores; marker genes; and basic QC / composition. Reads
`Data/PaperFigures_combo.h5ad` (run `00_Prepare_data.ipynb` first). All panels
are written to `figures/`.

In [ ]:
import _figutils as fu
import importlib; importlib.reload(fu)  # pick up edits to _figutils without kernel restart
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fu.set_theme()
adata = fu.load("processed")
fu.set_state_categories(adata)
adata.obs["time_point"] = pd.Categorical(
    adata.obs["time_point"].astype(str), categories=fu.TIMEPOINT_ORDER, ordered=True)
adata.obs["phase"] = adata.obs["phase"].astype("category")

state_colors = [fu.STATE_PALETTE[c] for c in adata.obs[fu.CELL_STATE_COL].cat.categories]
time_colors = [fu.TIME_PALETTE[c] for c in adata.obs["time_point"].cat.categories]
phase_colors = [fu.PHASE_PALETTE.get(c, "#999999") for c in adata.obs["phase"].cat.categories]
print(adata.shape, "|", adata.n_obs, "cells")

### a — UMAP coloured by cell state

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color=fu.CELL_STATE_COL, palette=state_colors, size=3,
           ax=ax, show=False, frameon=False, title="Cell state")
fu.savefig("overview_a_umap_cellstate", fig)

### b — UMAP coloured by time point

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color="time_point", palette=time_colors, size=3,
           ax=ax, show=False, frameon=False, title="Time point")
fu.savefig("overview_b_umap_timepoint", fig)

### c — UMAP coloured by cell-cycle phase

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color="phase", palette=phase_colors, size=3,
           ax=ax, show=False, frameon=False, title="Cell-cycle phase")
fu.savefig("overview_c_umap_phase", fig)

### d — Cell-state composition: control vs perturbed, by time point

For each time-point and cell state, the percentage of **control** (NTC-only) vs **perturbed** (≥ 1 target KO) cells in that state (grouped bars, each group's cells summing to 100% across states within a timepoint).

In [ ]:
from scipy.stats import fisher_exact; from statsmodels.stats.multitest import multipletests

order = fu.ordered_states(adata)

# Split cells: control = no target-gene guide (NTC-only); perturbed = >=1 target KO.
tgt = adata.obs[fu.TARGET_GENES].sum(axis=1).values
adata.obs["pert_group"] = np.where(tgt >= 1, "perturbed", "control")
print(adata.obs["pert_group"].value_counts(), "\n")

# Composition: for each (group, timepoint), the % of that group's cells in each state.
comp = {}
for g in ["control", "perturbed"]:
    for day in fu.TIMEPOINT_ORDER:
        sub = adata.obs[(adata.obs["pert_group"] == g) &
                        (adata.obs["time_point"].astype(str) == day)]
        comp[(g, day)] = (sub[fu.CELL_STATE_COL].value_counts()
                          .reindex(order).fillna(0) / max(len(sub), 1) * 100)

# Significance: Fisher perturbed-vs-control (in-state vs not) per (timepoint, state); BH over all.
gg = adata.obs["pert_group"].values
st = adata.obs[fu.CELL_STATE_COL].astype(str).values
tp = adata.obs["time_point"].astype(str).values
cells, pvals = [], []
for day in fu.TIMEPOINT_ORDER:
    pert = (tp == day) & (gg == "perturbed"); ctrl = (tp == day) & (gg == "control")
    for s in order:
        a = int((pert & (st == s)).sum()); b = int(pert.sum() - a)
        c = int((ctrl & (st == s)).sum()); d = int(ctrl.sum() - c)
        _, pv = fisher_exact([[a, b], [c, d]], alternative="two-sided")
        cells.append((day, s)); pvals.append(pv)
fdr_map = {cs: q for cs, q in zip(cells, multipletests(pvals, method="fdr_bh")[1])}

def _stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"

comp_df = pd.DataFrame({f"{g}_{day}": comp[(g, day)]
                        for g in ["control", "perturbed"] for day in fu.TIMEPOINT_ORDER})
comp_df.index.name = "state"
for day in fu.TIMEPOINT_ORDER:
    comp_df[f"fdr_{day}"] = [fdr_map[(day, s)] for s in order]
comp_df.to_csv(fu.FIG_DIR / "Fig1d_control_vs_perturbed_composition.csv")

# Grouped bars: per state, control vs perturbed; one panel per timepoint; * = perturbed-vs-control FDR.
GROUP_COLOR = {"control": "#9e9e9e", "perturbed": "#d62728"}
x = np.arange(len(order)); w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, day in zip(axes, fu.TIMEPOINT_ORDER):
    for i, g in enumerate(["control", "perturbed"]):
        ax.bar(x + (i - 0.5) * w, comp[(g, day)].values, width=w,
               color=GROUP_COLOR[g], label=g, edgecolor="k", lw=0.3)
    for xi, s in enumerate(order):
        top = max(comp[("control", day)].values[xi], comp[("perturbed", day)].values[xi])
        ax.text(x[xi], top + 1.0, _stars(fdr_map[(day, s)]), ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha="right")
    ax.set_title(day, fontsize=11); ax.set_xlabel("")
axes[0].set_ylabel("% of the group's cells in the state")
axes[1].legend(frameon=False, fontsize=9)
fig.suptitle("Cell-state composition: control vs perturbed cells, by time point\n"
             "(* perturbed-vs-control Fisher FDR; *<0.05, **<0.01, ***<0.001)", fontsize=10)
fu.savefig("overview_d_composition_by_time", fig)

### e — Trajectory signature scores on the UMAP

In [ ]:
score_cols = [c for c in fu.STATE_SCORE_COLS if c in adata.obs.columns]
sc.pl.umap(adata, color=score_cols, color_map="coolwarm", size=3,
           ncols=4, vmax="p99", vmin="p1", frameon=False, show=False)
fu.savefig("overview_e_umap_state_scores", plt.gcf())

### f — Marker / target-gene expression per cell state

In [ ]:
# State markers + the screened transcription factors (kept if present).
markers = ["ASCL1", "INSM1", "CHGA", "CHGB", "SYP", "NEUROD1", "NEUROG1",
           "KRT8", "KRT18", "EPCAM", "VIM", "TWIST1", "CDKN1A"]
markers = [m for m in markers if m in adata.raw.var_names]

# Several target genes (ASCL1, NEUROD1, ...) are ALSO one-hot guide columns in
# adata.obs, which makes scanpy's gene lookup ambiguous ("found in both
# adata.obs and adata.raw.var_names"). Hide those obs columns just for the plot,
# then restore them.
clash = [c for c in fu.GUIDE_COLS if c in adata.obs.columns]
saved = adata.obs[clash].copy()
adata.obs.drop(columns=clash, inplace=True)
try:
    sc.pl.dotplot(adata, markers, groupby=fu.CELL_STATE_COL,
                  categories_order=fu.ordered_states(adata),
                  use_raw=True, standard_scale="var", show=False)
    fu.savefig("overview_f_marker_dotplot", plt.gcf())
finally:
    adata.obs[clash] = saved

### QC — sequencing depth & gene complexity

Histogram of UMIs and genes detected per cell, each with a horizontal boxplot on top (box = IQR, line = median, diamond = mean, whiskers = 1.5×IQR).

In [ ]:
def hist_with_box(values, ax_box, ax_hist, color="#4C72B0", bins=80, logy=True):
    """Histogram (ax_hist) with a horizontal boxplot above it (ax_box, shared x).

    Box = IQR, central line = median, diamond = mean, whiskers = 1.5*IQR.
    """
    values = np.asarray(values, dtype=float)
    q1, med, q3 = np.percentile(values, [25, 50, 75])
    mean = values.mean()

    ax_hist.hist(values, bins=bins, color=color, edgecolor="none")
    if logy:
        ax_hist.set_yscale("log")
    ax_hist.axvline(med, color="k", ls="-", lw=1)
    ax_hist.axvline(mean, color="k", ls="--", lw=1)

    bp = ax_box.boxplot(values, vert=False, widths=0.6, showfliers=False,
                        showmeans=True, patch_artist=True,
                        meanprops=dict(marker="D", markerfacecolor="k",
                                       markeredgecolor="k", markersize=4),
                        medianprops=dict(color="k", lw=1.2),
                        whiskerprops=dict(color="0.4"), capprops=dict(color="0.4"))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.6)
    ax_box.set_yticks([])
    for sp in ax_box.spines.values():
        sp.set_visible(False)
    plt.setp(ax_box.get_xticklabels(), visible=False)

    txt = (f"n = {len(values):,}\nmedian = {med:,.0f}\nmean = {mean:,.0f}\n"
           f"IQR = {q1:,.0f}–{q3:,.0f}")
    ax_hist.text(0.97, 0.95, txt, transform=ax_hist.transAxes, ha="right", va="top",
                 fontsize=8, bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85))


fig = plt.figure(figsize=(8, 4))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 6], hspace=0.05, wspace=0.3)

axb0 = fig.add_subplot(gs[0, 0]); axh0 = fig.add_subplot(gs[1, 0], sharex=axb0)
hist_with_box(adata.obs["UMI_counts"], axb0, axh0, color="#4C72B0")
axh0.set_xlabel("UMIs / cell"); axh0.set_ylabel("Cells (log)")
axb0.set_title("Sequencing depth", fontsize=10)

axb1 = fig.add_subplot(gs[0, 1]); axh1 = fig.add_subplot(gs[1, 1], sharex=axb1)
hist_with_box(adata.obs["gene_number"], axb1, axh1, color="#55A868")
axh1.set_xlabel("Genes detected / cell"); axh1.set_ylabel("Cells (log)")
axb1.set_title("Gene complexity", fontsize=10)

fu.savefig("overview_QC_depth", fig)

### QC — additional panels

Library complexity, cell counts per time point / state / guide-complexity, sequencing depth per state, and cell-cycle phase composition.

In [ ]:
order = fu.ordered_states(adata)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

# 1) UMIs vs genes detected (library complexity).
ax = axes[0, 0]
hb = ax.hexbin(adata.obs["UMI_counts"], adata.obs["gene_number"],
               gridsize=60, bins="log", cmap="viridis")
ax.set_xlabel("UMIs / cell"); ax.set_ylabel("Genes detected")
ax.set_title("Library complexity")
fig.colorbar(hb, ax=ax, label="log10 cells")

# 2) Cells per time point.
ax = axes[0, 1]
n = adata.obs["time_point"].value_counts().reindex(fu.TIMEPOINT_ORDER)
ax.bar(n.index.astype(str), n.values,
       color=[fu.TIME_PALETTE[t] for t in n.index], edgecolor="k", lw=0.4, width=0.6)
ax.set_ylabel("Cells"); ax.set_title("Cells per time point")
for i, v in enumerate(n.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)

# 3) Single- vs double-KO cells per time point (with counts on each segment).
ax = axes[0, 2]
ct = (pd.crosstab(adata.obs["time_point"], adata.obs["N_genes_targeted"])
      .reindex(fu.TIMEPOINT_ORDER))
ngt_cols = list(ct.columns)
labels = [f"{c} guide" + ("s" if c > 1 else "") for c in ngt_cols]
colors = ["#bdbdbd", "#756bb1"]
xpos = np.arange(len(ct))
bottom = np.zeros(len(ct))
for j, c in enumerate(ngt_cols):
    vals = ct[c].values
    ax.bar(xpos, vals, bottom=bottom, color=colors[j % len(colors)],
           edgecolor="k", lw=0.3, width=0.6, label=labels[j])
    for i, v in enumerate(vals):
        ax.text(xpos[i], bottom[i] + v / 2, f"{v:,}", ha="center", va="center",
                fontsize=8)
    bottom += vals
for i, tot in enumerate(bottom):            # total on top of each bar
    ax.text(xpos[i], tot, f"{int(tot):,}", ha="center", va="bottom",
            fontsize=8, fontweight="bold")
ax.set_xticks(xpos); ax.set_xticklabels(ct.index.astype(str))
ax.set_xlabel(""); ax.set_ylabel("Cells"); ax.set_title("Guide complexity", pad=24)
ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0),
          frameon=False, fontsize=8)

# 4) Cells per state.
ax = axes[1, 0]
n = adata.obs[fu.CELL_STATE_COL].value_counts().reindex(order)
ax.bar(range(len(n)), n.values,
       color=[fu.STATE_PALETTE[s] for s in n.index], edgecolor="k", lw=0.4)
ax.set_xticks(range(len(n))); ax.set_xticklabels(n.index, rotation=30, ha="right")
ax.set_ylabel("Cells"); ax.set_title("Cells per state")

# 5) Sequencing depth per state (check depth doesn't track state).
ax = axes[1, 1]
sns.violinplot(data=adata.obs, x=fu.CELL_STATE_COL, y="UMI_counts", order=order,
               palette=[fu.STATE_PALETTE[s] for s in order], cut=0, inner="box", ax=ax)
ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_xlabel(""); ax.set_ylabel("UMIs / cell"); ax.set_title("Depth by state")

# 6) Cell-cycle phase composition per state, split by time point.
#    For each state, two stacked bars: left = day04, right = day10.
ax = axes[1, 2]
phase_cats = [p for p in ["G1", "S", "G2M"] if p in adata.obs["phase"].unique().tolist()]
x = np.arange(len(order))
w = 0.38
for tp, off in zip(fu.TIMEPOINT_ORDER, [-0.20, 0.20]):
    sub = adata.obs[adata.obs["time_point"] == tp]
    ph = (pd.crosstab(sub[fu.CELL_STATE_COL], sub["phase"], normalize="index")
          .reindex(order)[phase_cats])
    bottom = np.zeros(len(order))
    for p in phase_cats:
        ax.bar(x + off, ph[p].values, width=w, bottom=bottom,
               color=fu.PHASE_PALETTE.get(p, "#999999"), edgecolor="white", lw=0.3,
               label=(p if off < 0 else None))
        bottom += ph[p].values
ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_ylabel("Fraction of cells")
ax.set_xlabel("left = day04   |   right = day10", fontsize=8)
ax.set_ylim(0, 1)
ax.legend(ncol=len(phase_cats), loc="lower center", bbox_to_anchor=(0.5, 1.01),
          frameon=False, fontsize=8, title="Cell-cycle phase by state",
          title_fontsize=9)

fu.savefig("overview_QC_panels", fig)

### QC — cells per perturbation

Number of cells recovered for each perturbation (`perturbation_clean`, i.e. single-KO `X+NTC` collapsed to `X`). Bars coloured by control / single KO / double KO and sorted by abundance.

In [ ]:
import matplotlib.patches as mpatches

# Cells per perturbation, split into day04 / day10 (shared x-order = total count).
ct = pd.crosstab(adata.obs["perturbation_clean"], adata.obs["time_point"])
order = ct.sum(axis=1).sort_values(ascending=False).index
ct = ct.reindex(order)[fu.TIMEPOINT_ORDER]

def _kind(p):
    if p == fu.NTC_LABEL:
        return "control"
    return "single KO" if "+" not in p else "double KO"

kind_color = {"control": "#000000", "single KO": "#4C72B0", "double KO": "#DD8452"}
colors = [kind_color[_kind(p)] for p in order]
ymax = ct.values.max() * 1.4

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
for ax, tp in zip(axes, fu.TIMEPOINT_ORDER):
    ax.bar(range(len(order)), ct[tp].values, color=colors, width=0.85, edgecolor="none")
    ax.set_yscale("log")
    ax.set_ylim(10, ymax)
    ax.set_ylabel("Cells (log)")
    ax.set_title(tp, loc="left", fontsize=11, fontweight="bold")

axes[-1].set_xticks(range(len(order)))
axes[-1].set_xticklabels(order, rotation=90, fontsize=10)
axes[-1].set_xlim(-0.7, len(order) - 0.3)
axes[-1].set_xlabel("Perturbation")
axes[0].legend(handles=[mpatches.Patch(color=kind_color[k], label=k)
                        for k in ["control", "single KO", "double KO"]],
               frameon=False, fontsize=8, ncol=3, loc="upper right")
fig.suptitle(f"Cells per perturbation by time point (n = {len(order)} perturbations)",
             fontsize=12)

ct.rename_axis("perturbation").to_csv(fu.FIG_DIR / "Fig1_cells_per_perturbation.csv")
fu.savefig("overview_QC_cells_per_perturbation", fig)